## PURPOSE: Running current classification model on random attachments, to see if we have any FN or TN

In [1]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from python_utilities.db_connection import DbConnection

In [2]:
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

INFO [2025-10-15 14:09:28] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


#### Lets read attachments that are sent us in previous 2 months

In [ ]:
past_ndays=1
today = datetime.now().date()
start_date = datetime.combine(today - timedelta(days=past_ndays), datetime.min.time())
end_date = datetime.combine(today, datetime.min.time())
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"Fetching attachment data for last {past_ndays} days")

In [ ]:
query = """

SELECT
    a.attachment_id,
    a.zendesk_id,
    a.comment_id,
    a.file_name,
    a.file_extension,
    tj.s3_link 			  AS textract_s3_link,
    tj.job_id             AS textract_job_id,
    tj.status             AS textract_status,
    tj.created_at         AS textract_created_at,
    zt.status             AS ticket_status,
    zt.origin             AS ticket_origin,
    ap.id                 AS prediction_db_id,
    ap.model_name,
    ap.type,
    ap.subtype,
    ap.value              AS prediction_value
FROM llm_zendesk_attachments AS a
JOIN textract_jobs          AS tj
  ON tj.attachment_id = a.attachment_id
JOIN llm_zendesk_tickets    AS zt
  ON zt.zendesk_id = a.zendesk_id
 AND (
      zt.comment_id = a.comment_id
      OR (zt.comment_id IS NULL AND a.comment_id IS NULL)
     )
LEFT JOIN llm_attachments_predictions AS ap
  ON ap.attachment_id = a.attachment_id
WHERE tj.status = 'SUCCEEDED'
AND tj.created_at >= NOW() - INTERVAL 1 DAY
AND zt.status = 'processed'
ORDER BY tj.created_at DESC
"""

#data = analytics_db.sql_to_df(query)

### Result of Above Query Will be read below. 

In [12]:
data = pd.read_csv('data/ATTACHMENT_sent_with_preds.csv')

In [ ]:
data

### Not get Attachment's Parsed Texts

In [ ]:
# drop null values
data = data.dropna(subset=['textract_s3_link'])
print(data.shape)

In [ ]:
# get the texts from textract_s3_link
s3_links_sampled = np.random.choice(data['textract_s3_link'].unique(), size=15000, replace=False).tolist()
print(len(s3_links_sampled))

In [ ]:
import boto3
textract_texts = {}
def get_jsons_from_s3(s3_links):
    """
    Retrieves JSON objects from given S3 addresses.
    """
    global textract_texts
    
    session = boto3.Session(profile_name='739275445236_DataScienceUser')
    s3_client = session.client('s3')

    index = 0
    for link in s3_links:
        json_object = [{"BlockType": "LINE", "Text": ""}]
        try:
            # Parse the S3 URI
            parts = link.replace("s3://", "").split("/")
            bucket_name = parts[0]
            key = "/".join(parts[1:])
            # Get the object from S3
            response = s3_client.get_object(Bucket=bucket_name, Key=key)

            # Read the content and parse JSON
            content = response['Body'].read().decode('utf-8')
            json_object = json.loads(content)
            textract_texts[link] = json_object
        except Exception as e:
            print(f"Error from {link} {e}")
            textract_texts[link] = []
        print(f"Processed {index+1}/{len(s3_links)}: {link}")
        index += 1
        
        if index % 5000 == 0:
            print(f"Checkpoint at {index} links processed.")
            # Save intermediate results to avoid data loss
            with open('data/textract_texts_intermediate.json', 'w') as f:
                json.dump(textract_texts, f)
        
    return textract_texts

s3_link_json_pairs = get_jsons_from_s3(s3_links_sampled)

In [ ]:
s3_link_json_pairs

In [ ]:
s3_link_text_pairs = {link: "\n".join([d["Text"] for d in doc if d["BlockType"] == "LINE"]) for link, doc in s3_link_json_pairs.items()}
# def get_texts_from_textract_outputs(
#         textract_jsons):
#         return [
#             "\n".join([d["Text"] for d in doc if d["BlockType"] == "LINE"])
#             for doc in textract_jsons
#        ]
# texts = get_texts_from_textract_outputs(s3_link_json_pairs)

In [ ]:
s3_link_text_pairs

In [ ]:
# save s3_link_text_pairs
with open('data/s3_link_text_pairs.json', 'w') as f:
    json.dump(s3_link_text_pairs, f)

## Continue here

In [3]:
# read s3_link_text_pairs
with open('data/s3_link_text_pairs.json', 'r') as f:
    s3_link_text_pairs = json.load(f)

#### Import pipelines from src code

In [4]:
import os
print(os.getcwd())
os.chdir('../../')
print(os.getcwd())

/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court
/Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [5]:
from pathlib import Path

from src.domain.base.blueprints import AfterCourtModelBlueprint
from src.domain.attachments.attachments_entities import Attachment, CleanTexts
from src.services.attachment_processing.base_input_processor import AfterCourtAttachmentPreprocessor
from src.services.models.aftercourt_classification_model import AfterCourtClassificationModel
from src.db.database_manager import DatabaseManager
from src.services.secrets_manager import SecretsManager
from src.prod_repository.raw_zendesk_tickets_repository import RawZendeskTicketRepository

In [ ]:


models_root = Path.cwd()
aftercourt_config = AfterCourtModelBlueprint.from_json(
    str(models_root / "configs" / "models" / "after_court" / "aftercourt_classification.json")
)._update_pathes(str(models_root))

s3_text_items = list(s3_link_text_pairs.items())
attachments_for_inference = [
    Attachment(
        attachment_id=str(idx),
        created_at=datetime.utcnow(),
        status="processed_by_textract",
        clean_text=CleanTexts(original=text),
    )
    for idx, (_, text) in enumerate(s3_text_items)
]

aftercourt_preprocessor = AfterCourtAttachmentPreprocessor(
    name=aftercourt_config.name,
    preprocessing_config=aftercourt_config.preprocessing,
)
attachments_preprocessed = aftercourt_preprocessor.preprocess(attachments_for_inference)

aftercourt_model = AfterCourtClassificationModel(
    config=aftercourt_config.model,
    name=aftercourt_config.name,
 )

attachments_with_aftercourt_preds = aftercourt_model.predict_attachments(
    attachments_preprocessed
)

aftercourt_predictions = pd.DataFrame(
    [
        {
            "s3_link": link,
            **(attachment.models_outputs.aftercourt_classification or {}),
        }
        for (link, _), attachment in zip(s3_text_items, attachments_with_aftercourt_preds)
    ]
)

aftercourt_predictions.head()

In [ ]:
aftercourt_predictions

### Continue here

In [6]:
os.chdir('/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court')
#aftercourt_predictions.to_csv('data/aftercourt_predictions_on_random_attachments.csv', index=False)
aftercourt_predictions = pd.read_csv('data/aftercourt_predictions_on_random_attachments.csv')

In [7]:
ladung_documents = aftercourt_predictions[aftercourt_predictions['is_aftercourt'] == True]

In [8]:
ladung_documents

,s3_link,is_aftercourt,aftercourt_type
267,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
524,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
1308,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
1584,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
1659,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
1709,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
1938,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
2334,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
2695,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung
2797,s3://pair-data-engineering-new/ocr_prepared_ou...,True,ladung


In [9]:
total_documnents = len(s3_link_text_pairs)
ladung_documents_count = ladung_documents.shape[0]
ladung_documents_percentage = (ladung_documents_count / total_documnents) * 100
print(f"Total Documents: {total_documnents}")
print(f"Ladung Documents: {ladung_documents_count}")
print(f"Ladung Documents Percentage: {ladung_documents_percentage:.2f}%")

Total Documents: 15000
Ladung Documents: 59
Ladung Documents Percentage: 0.39%


In [10]:
s3_links_to_get = ladung_documents['s3_link'].tolist()

In [13]:
ladung_documents_all_data = data[data['textract_s3_link'].isin(s3_links_to_get)]

In [14]:
# add textract_text column to ladung_documents_all_data
ladung_documents_all_data['textract_text'] = ladung_documents_all_data['textract_s3_link'].map(s3_link_text_pairs)

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_93067/2522587128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ladung_documents_all_data['textract_text'] = ladung_documents_all_data['textract_s3_link'].map(s3_link_text_pairs)


In [15]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,file_name,file_extension,textract_s3_link,textract_job_id,textract_status,textract_created_at,ticket_status,ticket_origin,prediction_db_id,model_name,type,subtype,prediction_value,textract_text
5614,16982887-1,16982887,NaN,Dokument_145325_13102025_165430.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,dc12ff30de567e64eb06ccd6aaf8f931f3b60344d1f8fe...,SUCCEEDED,2025-10-13 15:16:14,processed,DE,1306685.0,payment_proof,payment_proof,is_payment_proof,'False',Stefanie Herwegh\nBahnhofstraße 24\nObergerich...
5615,16982887-1,16982887,NaN,Dokument_145325_13102025_165430.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,dc12ff30de567e64eb06ccd6aaf8f931f3b60344d1f8fe...,SUCCEEDED,2025-10-13 15:16:14,processed,DE,1306686.0,payment_proof,payment_proof,payment_proof_type,'none',Stefanie Herwegh\nBahnhofstraße 24\nObergerich...
5616,16982887-1,16982887,NaN,Dokument_145325_13102025_165430.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,dc12ff30de567e64eb06ccd6aaf8f931f3b60344d1f8fe...,SUCCEEDED,2025-10-13 15:16:14,processed,DE,1306687.0,multi_attachment,multi_attachment,attachment_type,'other',Stefanie Herwegh\nBahnhofstraße 24\nObergerich...
16229,16978019-1,16978019,NaN,Anlage131025081551.PDF,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,1619c65869115ecb60504c8ddc176be64ef48b34b69cf9...,SUCCEEDED,2025-10-13 07:14:40,processed,DE,1296223.0,payment_proof,payment_proof,is_payment_proof,'False',Gerichtsvollzieher Heuß\nRietstraße 4 / 74740 ...
16230,16978019-1,16978019,NaN,Anlage131025081551.PDF,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,1619c65869115ecb60504c8ddc176be64ef48b34b69cf9...,SUCCEEDED,2025-10-13 07:14:40,processed,DE,1296224.0,payment_proof,payment_proof,payment_proof_type,'none',Gerichtsvollzieher Heuß\nRietstraße 4 / 74740 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655467,16645997-1,16645997,NaN,Dokument_76525_18082025_090116.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,cc3d38195855da08164938ea4635f0dfe68e8a093a5894...,SUCCEEDED,2025-08-18 08:07:37,processed,DE,738331.0,invalid_payment_proof,invalid_payment_proof,attachment_type,'other',M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...
656307,16645886-1,16645886,NaN,Dokument_77025_18082025_084727.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,014defaa6e042f7923991dc01ef3ff736e45bde8df96af...,SUCCEEDED,2025-08-18 07:09:59,processed,DE,738047.0,bank_statement,bank_statement,NaN,'False',M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...
656308,16645886-1,16645886,NaN,Dokument_77025_18082025_084727.pdf,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,014defaa6e042f7923991dc01ef3ff736e45bde8df96af...,SUCCEEDED,2025-08-18 07:09:59,processed,DE,738048.0,invalid_payment_proof,invalid_payment_proof,attachment_type,'other',M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...
666365,16638303-1,16638303,NaN,DR-II+098125+Nachr+Gl.Nr.+vom+Termin+15.08.202...,pdf,s3://pair-data-engineering-new/ocr_prepared_ou...,57190ced47f9a5995f363954c17b6be64fdc800c2e9a8a...,SUCCEEDED,2025-08-15 12:15:54,processed,DE,728100.0,bank_statement,bank_statement,NaN,'False',Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...


In [16]:
ladung_documents_all_data = ladung_documents_all_data.drop_duplicates(subset=['attachment_id', 'model_name', 'subtype'])

In [17]:
ladung_documents_all_data["pivot_column_names"] = np.where(
    ladung_documents_all_data["subtype"].isna(),
    ladung_documents_all_data["model_name"],
    ladung_documents_all_data["model_name"] + '.' + ladung_documents_all_data["subtype"].astype(str)
)

ladung_documents_all_data.dropna(subset=['pivot_column_names'], inplace=True)

ladung_documents_all_data = ladung_documents_all_data.pivot(
    index=["attachment_id", 'zendesk_id','comment_id','textract_text'],
    columns='pivot_column_names',
    values="prediction_value"
).reset_index()

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_93067/1995249942.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ladung_documents_all_data["pivot_column_names"] = np.where(
/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_93067/1995249942.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ladung_documents_all_data.dropna(subset=['pivot_column_names'], inplace=True)


In [18]:
ladung_documents_all_data

pivot_column_names,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,payment_proof.payment_proof_type,payment_proof.recipient,payment_proof.recipient_iban
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, R.504\nObergerich...",NaN,NaN,NaN,'False',NaN,NaN,'none',NaN,NaN


In [19]:
zendesk_ids_to_get = tuple(ladung_documents_all_data.zendesk_id.astype(int).astype(str).tolist())
all_text = pd.DataFrame()
for origin in ['DE','NL','CH','AT', 'SE']:
    query = f"""
        SELECT
            CAST(rzt.zendesk_ticket_id AS INT) AS zendesk_id,
            rzt.content as text,
            rzt.from_email as sender_email,
            rzt.to_email as receiver_email
        FROM 
            raw_zendesk_tickets as rzt
        WHERE 
            rzt.zendesk_ticket_id in {zendesk_ids_to_get}
    """
    psql_db = DbConnection('PROD', f'PROD_{origin}')
    tmp = psql_db.sql_to_df(query)
    tmp['origin']=origin
    all_text = pd.concat([all_text,tmp])

INFO [2025-10-15 14:10:23] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2025-10-15 14:10:24] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2025-10-15 14:10:24] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2025-10-15 14:10:25] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2025-10-15 14:10:25] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [20]:
all_text

,zendesk_id,text,sender_email,receiver_email,origin
0,16645886,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE
1,16645997,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE
2,16650437,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE
3,16657367,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE
4,16666355,Sache 1 DR 858/25 E.ON Energie Deutschland Gmb...,hgv-patricia-debus@gvzpost.de,aftercourt@pairfinance.de,DE
5,16671633,"Ihr Zeichen: 159641359520, Mein Zeichen: DR II...",l.sontheimer@gvjustiz.de,verfahren@pairfinance.de,DE
6,16685127,"Ihr Zeichen: 1066011550570, Mein Zeichen: DR I...",r.kindelbacher@gvzentrale.de,aftercourt@pairfinance.de,DE
7,16686089,Sache 616 DR II 362/25 mydays GmbH ./. Koca\nA...,gerichtsvollzieher-kultscher@gvpost.de,verfahren@pairfinance.de,DE
8,16701287,Sache DR 722/25 Liquandum Capital GmbH ./. Fuc...,s.landgraf@gvzpost.de,aftercourt@pairfinance.de,DE
9,16708455,"DR-II 0987/25, Liquandum Capital GmbH ./. Yesi...",herbert.blomeyer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE


In [21]:
ladung_documents_all_data = ladung_documents_all_data.merge(
    all_text,
    how='left',
    on='zendesk_id',
    suffixes=('', '_ticket'))

In [22]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,payment_proof.payment_proof_type,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, R.504\nObergerich...",NaN,NaN,NaN,'False',NaN,NaN,'none',NaN,NaN,"Ihr Zeichen: 171931616477, Mein Zeichen: DR II...",ogvinluedke@gvzentrale.de,aftercourt@pairfinance.de,DE


### Apply cleaning to ticket text

In [23]:
from src.services.cleaning.detailed_text_cleaner import DetailedTextCleaner
cleaner = DetailedTextCleaner()
ladung_documents_all_data['cleaned_text'] = ladung_documents_all_data['text'].apply(lambda x: cleaner.clean_text(x))

In [24]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,payment_proof.payment_proof_type,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin,cleaned_text
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Zwang..."
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten..."
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten..."
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach..."
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach..."
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Anlag..."
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE,Sehr geehrte Damen und Herren\n\nIn der Zwangs...
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach..."
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten..."
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, R.504\nObergerich...",NaN,NaN,NaN,'False',NaN,NaN,'none',NaN,NaN,"Ihr Zeichen: 171931616477, Mein Zeichen: DR II...",ogvinluedke@gvzentrale.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten..."


### Translate Ticket text

In [25]:
import boto3
import awswrangler.secretsmanager as sm
import boto3
from dataclasses import dataclass
import deepl
from loguru import logger
import time


@dataclass
class SecretsManagerBlueprint:
    secret_id: str = "ds-llm-production"
    region_name: str = "eu-central-1"

def get_secret(secret_id: str, value: str, aws_region: str) -> str:
    return sm.get_secret_json(
        secret_id,
        boto3_session=boto3.Session(region_name=aws_region),
    ).get(value)


class TranslationService:
    def __init__(self):
        self.secrets_manager = SecretsManagerBlueprint()
        self._translator = deepl.Translator(get_secret(
            self.secrets_manager.secret_id,
            "deepl_token",
            self.secrets_manager.region_name
        )
        )

    def translate(self, text, target_lang="EN-GB"):
        try:
            return self._translator.translate_text(text, target_lang=target_lang).text
            # return text
        except Exception as e:
            logger.error(f'Error in translate text: {text}')
            logger.opt(exception=True).error(str(e))
            return text
        
tr = TranslationService()

INFO [2025-10-15 14:10:29] - Found credentials in shared credentials file: ~/.aws/credentials


In [26]:
ladung_documents_all_data['translated_text'] = ladung_documents_all_data['cleaned_text'].apply(lambda x: tr.translate(x, target_lang="EN-GB") if pd.notnull(x) and len(x.strip()) > 0 else x)

INFO [2025-10-15 14:10:29] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:29] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:30] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:30] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:10:30] - DeepL API response status_code=200 url=https://api.dee

In [53]:
ladung_documents_all_data['translated_text_TEXTRACT'] = ladung_documents_all_data['textract_text'].apply(lambda x: tr.translate(x, target_lang="EN-GB") if pd.notnull(x) and len(x.strip()) > 0 else x)

INFO [2025-10-15 14:39:23] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:23] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:23] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:23] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:23] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:24] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:24] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:24] - DeepL API response status_code=200 url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:24] - Request to DeepL API method=POST url=https://api.deepl.com/v2/translate
INFO [2025-10-15 14:39:24] - DeepL API response status_code=200 url=https://api.dee

In [27]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,payment_proof.payment_proof_type,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin,cleaned_text,translated_text
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Zwang...","Ladies and Gentlemen,\n\nin the enforcement pr..."
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Anlag...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE,Sehr geehrte Damen und Herren\n\nIn der Zwangs...,Dear Sir or Madam\n\nIn the enforcement procee...
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ..."
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, R.504\nObergerich...",NaN,NaN,NaN,'False',NaN,NaN,'none',NaN,NaN,"Ihr Zeichen: 171931616477, Mein Zeichen: DR II...",ogvinluedke@gvzentrale.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ..."


### Lets check how we sent them to the ccenter

In [28]:
zendesk_ids_to_get = tuple(ladung_documents_all_data.zendesk_id.tolist())

query = f"""
    SELECT
        ci.zendesk_id,
        ci.comment_id,
        ci.intents,
        ci.ready_for_automation,
        ci.created_at
    FROM 
        ccenter_intents ci
    WHERE 
        ci.zendesk_id in {zendesk_ids_to_get}
"""


ccenter_intents_df = analytics_db.sql_to_df(query)

In [29]:
ccenter_intents_df

,zendesk_id,comment_id,intents,ready_for_automation,created_at
0,16638303,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-15 14:48:39
1,16645886,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 09:16:00
2,16645997,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 11:25:53
3,16650437,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 22:05:08
4,16654322,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 08:51:59
5,16654606,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 10:51:17
6,16656735,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 16:28:03
7,16656958,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 16:28:17
8,16657367,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 20:21:30
9,16664086,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-20 14:29:36


In [30]:
ccenter_intents_df

,zendesk_id,comment_id,intents,ready_for_automation,created_at
0,16638303,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-15 14:48:39
1,16645886,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 09:16:00
2,16645997,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 11:25:53
3,16650437,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-18 22:05:08
4,16654322,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 08:51:59
5,16654606,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 10:51:17
6,16656735,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 16:28:03
7,16656958,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 16:28:17
8,16657367,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-19 20:21:30
9,16664086,NaN,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0,2025-08-20 14:29:36


In [31]:
ladung_documents_all_data = ladung_documents_all_data.merge(
    ccenter_intents_df.drop(columns=['created_at']),
    how='left',
    on=['zendesk_id','comment_id'],
    suffixes=('', '_ccenter'))

In [32]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,...,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin,cleaned_text,translated_text,intents,ready_for_automation
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Zwang...","Ladies and Gentlemen,\n\nin the enforcement pr...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Anlag...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE,Sehr geehrte Damen und Herren\n\nIn der Zwangs...,Dear Sir or Madam\n\nIn the enforcement procee...,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, 

In [2]:
#ladung_documents_all_data.to_csv('data/ladung_documents_all_data.csv', index=False)
ladung_documents_all_data = pd.read_csv('data/ladung_documents_all_data.csv')

### Continue here

In [34]:
# lets check the results to understand how ladung documents were sent to us

In [35]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,...,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin,cleaned_text,translated_text,intents,ready_for_automation
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Zwang...","Ladies and Gentlemen,\n\nin the enforcement pr...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Anlag...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE,Sehr geehrte Damen und Herren\n\nIn der Zwangs...,Dear Sir or Madam\n\nIn the enforcement procee...,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, 

In [3]:
import os
print(os.getcwd())
os.chdir('../../')
print(os.getcwd())

/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court
/Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [4]:
from IPython.display import display, HTML

def show_colored_params(params, key_color="#ff6b6b", value_color="#4ecdc4"):
    if not params:
        display(
            HTML(
                "<div style='font-family:monospace;background:#1e1e1e;padding:12px;border-radius:8px;color:#ff6b6b;'>No parameters extracted.</div>"
            )
        )
        return

    content = "".join(
        f"<div><span style='color:{key_color};font-weight:600;'>{key}</span>: "
        f"<span style='color:{value_color};'>{value}</span></div>" for key, value in params.items()
    )
    display(
        HTML(
            "<div style='font-family:monospace;background:#1e1e1e;padding:12px;border-radius:8px;'>"
            + content
            + "</div>"
        )
    )
    
def print_colored_text(text: str, strings: list[str]) -> None:
    text = text.lower()
    colors = [
        "\033[91m",  # red
        "\033[92m",  # green
        "\033[94m",  # blue
        "\033[95m",  # magenta
        "\033[96m",  # cyan
        "\033[93m",  # yellow
    ]
    reset = "\033[0m"
    colored_text = text
    for idx, fragment in enumerate(strings):
        if not fragment:
            continue
        color = colors[idx % len(colors)]
        colored_text = colored_text.replace(fragment, f"{color}{fragment}{reset}")
    print(colored_text)

In [5]:
from src.services.attachment_processing.aftercourt_extractors.ladung.strategy import LadungParameterExtractorStrategy
extractor = LadungParameterExtractorStrategy()

In [6]:
ladung_documents_all_data

,attachment_id,zendesk_id,comment_id,textract_text,bank_statement,invalid_payment_proof.attachment_type,multi_attachment.attachment_type,payment_proof.is_payment_proof,payment_proof.payment_amount,payment_proof.payment_date,...,payment_proof.recipient,payment_proof.recipient_iban,text,sender_email,receiver_email,origin,cleaned_text,translated_text,intents,ready_for_automation
0,16638303-1,16638303,NaN,Bindlacher Str. 3\nMarkus Rödel\n95448 Bayreut...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 0981/25, Liquandum Capital GmbH ./. Scha...",markus-roedel@gerichtsvollzieher.de,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Zwang...","Ladies and Gentlemen,\n\nin the enforcement pr...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
1,16645886-1,16645886,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 173354132123, Mein Zeichen: DR II...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
2,16645997-1,16645997,NaN,M. Suttmann\nGerichtsvollzieherin\nJoseph-von-...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 17338057404, Mein Zeichen: DR II ...",melanie.suttmann@gmail.com,verfahren@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
3,16650437-1,16650437,NaN,Gerichtsvollzieherin\nBorsigallee 10\nL. Riedl...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Sache DR II 1774/25, Ihr Aktz. 112275389636\nA...",linda.riedl@ag-bonn.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
4,16654322-1,16654322,NaN,Obergerichtsvollzieherin\nEuropaplatz 11\nBeat...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache DR II 1058/25 PAIR Finance GmbH ./. Habi...,beate.borren@ag-moenchengladbach.nrw.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
5,16654606-1,16654606,NaN,H. Schuller\nHauptgerichtsvollzieher\nLechstr....,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 46 DR 1101/25 Liquandum Capitel II GmbH ...,h.schuller@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nin der Anlag...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
6,16656735-1,16656735,NaN,A. Hornauer\nAugust-Fritzsche-Straße 10\n04838...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"DR-II 1054/25, N26 Bank SE vertr.d.d. Vorstand...",hornauer@gerichtsvollzieher.de,aftercourt@pairfinance.de,DE,Sehr geehrte Damen und Herren\n\nIn der Zwangs...,Dear Sir or Madam\n\nIn the enforcement procee...,"[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
7,16656958-1,16656958,NaN,Gerichtsvollzieher Marc Oedekoven\nAmtsgericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,Sache 31 DR II 1052/25 Liquandum Capital II Gm...,marc.oedekoven@gvzpost.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nzu o.g. Sach...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
8,16657367-1,16657367,NaN,M. Krammer\nGabelsbergerstraße 52\nObergericht...,'False','other',NaN,NaN,NaN,NaN,...,NaN,NaN,"Ihr Zeichen: 171935511138, Mein Zeichen: DR II...",gv-krammer@online.de,aftercourt@pairfinance.de,DE,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ladies and Gentlemen,\n\nPlease find enclosed ...","[{""params"": {}, ""language"": ""de"", ""ticket_id"":...",0.0
9,16664086-1,16664086,NaN,"Lüdke\nHellersdorfer Weg 35, 

In [ ]:
# MISCLASSIFIED DOCUMENT TEXTS:
# ALSO CHECK: Projects/after-court/document_quenstions
# 1 -> probably should be enforcement order
# 5 -> what should be this class?
# 27 - > we have a mail from tech@pairfinance.com with Email receiver:  None, the content is: GVZ - Scanned Brief
# Detected Slugs:

#     163100638366
# What is that? (This question is for index 27) -> btw the attachment looks like enforcement order
# 28 -> not ladung document, what is that? (might be enforcement order)
# 29 -> not ladung document, what is that? (might be enforcement order)
# 30 -> not ladung document, what is that? (might be enforcement order)
# 37 -> again looks like enforcement order
# 47-> again enforcement order
# 45 -> what is that?
# 53 -> interesting e-mail


In [10]:

index = 4
textract_text = ladung_documents_all_data.iloc[index]["textract_text"]
#textract_text_translated = ladung_documents_all_data.iloc[index]["translated_text_TEXTRACT"]
translated_text = ladung_documents_all_data.iloc[index]["translated_text"]
extracted_params = extractor.extract_aftercourt_parameters(textract_text)
slug = extracted_params.get("slug", "")
name, surname = extracted_params.get('debtor_name',' ').split(' ', 1) if 'debtor_name' in extracted_params else ('', '')
name = name.lower()
surname = surname.lower()
date = extracted_params.get('judicial_summon_date', '')
date = date.replace('-','.')
show_colored_params(extracted_params)
print("______________ATTACHMENT TEXT TRANSLATED______________")
print_colored_text(textract_text, [slug, name, surname, date])
print("_"*100)
print("______________EMAIL TICKET TEXT TRANSLATED______________")
print(translated_text)
print("_"*100)
print("Email sender: ", ladung_documents_all_data.iloc[index]['sender_email'])
print("Email receiver: ", ladung_documents_all_data.iloc[index]['receiver_email'])

______________ATTACHMENT TEXT TRANSLATED______________
obergerichtsvollzieherin
europaplatz 11
beate borren
41061 mönchengladbach
amtsgericht mönchengladbach
bürozeiten
montag + mittwoch
telefon
12.00 13.00 uhr
02161 9373390
dienstkonto
iban de36 3706 9252 8001 6890 20
bic genoded1ere
abs.: ogvin beate borren, europaplatz 11, 41061 mönchengla
e-mail
beate.borren@ag-moenchengladbach.nrw.de
pair finance gmbh
knesebeckstraße 62-63
10719 berlin
mein zeichen
ihr zeichen
dr ii 1058/25
159908222163
mönchengladbach, 19.08.2025
bitte immer angeben!
zwangsvollstreckungssache
pair finance gmbh, knesebeckstraße 62-63, 10719 berlin, tel. 030/340 602 950, e-mail
aftercourt@pairfinance.de
gegen
frau celina habiciak, flachsbleiche 104, 41179 mönchengladbach
sehr geehrte damen und herren,
in vorstehender angelegenheit habe ich gemäß ihrem auftrag termin zur abgabe der vermögensauskunft
auf
montag, 08.09.25, 09:40 uhr, europaplatz 11, 41061 mönchengladbach (hbf.)
bestimmt.
die erneute ladung war notwend